In [0]:
import numpy as np
from sklearn.manifold import MDS, LocallyLinearEmbedding, TSNE
import sys
sys.path.append('..')
from src.visualization import plot_dim_reduction

In [0]:
# Load clustered results from gold layer
df_label = spark.table("scientific_trend_analysis.rep.arxiv_clustered").toPandas()
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_km'].unique())} clusters")
display(df_label.head())

In [0]:
# Load LSA features from volume
import pickle
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features.pkl', 'rb') as f:
    lsa_result = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_km = df_label['Cluster_ID_km'].values

print(f"Loaded LSA features: {lsa_result.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample = df_label.groupby('Cluster_ID_km') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_km'].value_counts(normalize=True))

stratified_indices = stratified_sample.index.get_level_values(1).values
lsa_stratified = lsa_result[stratified_indices]

# Stratified samples
cluster_labels_numpy = np.array(cluster_labels_km)
km_sample_labels = cluster_labels_numpy[stratified_indices]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified.shape}")

In [0]:
# Run MDS on stratified LSA sample
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1, 
    max_iter=100
)
mds_2d = mds.fit_transform(lsa_stratified)
df_plot_mds = plot_dim_reduction(mds_2d, km_sample_labels, 'MDS', 'Cluster_ID_km', sample_size)

In [0]:
# Use stratified sample for LLE visualization
lle = LocallyLinearEmbedding(
    n_components=2, 
    random_state=2, 
    n_neighbors=8
)
lle_2d = lle.fit_transform(lsa_stratified)
df_plot_lle = plot_dim_reduction(lle_2d, km_sample_labels, 'LLE', 'Cluster_ID_km', sample_size)

In [0]:
# Use stratified sample for t-SNE visualization
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=30,
    init='pca'
)
tsne_2d = tsne.fit_transform(lsa_stratified)
df_plot_tsne = plot_dim_reduction(tsne_2d, km_sample_labels, 't-SNE', 'Cluster_ID_km', sample_size)